In [1]:
import pandas as pd
import itertools


In [4]:


files = {
	# DeBERTa
	"deberta_42":     rf"deberta_processed/deberta_seed_42_processed/submission_seed42.csv",
	"deberta_1337":   rf"deberta_processed/deberta_seed_1337_processed/submission_seed_daberta_processed1337.csv",
	"deberta_2024":   rf"deberta_processed/deberta_seed_2024_processed/submission_seed_daberta_processed2024.csv",
	"deberta_noseed": rf"deberta_processed/submission_deberta_processed_noseed.csv",

	# RoBERTa
	"roberta_42":     rf"roberta_processed/roberta_processed_seed_42/submission_seed_roberta_processed42.csv",
	"roberta_1337":   rf"roberta_processed/roberta_processed_seed_1337/submission_seed_roberta_processed1337.csv",
	"roberta_2024":   rf"roberta_processed/roberta_processed_seed_2024/submission_seed_roberta_processed2024.csv",
	"roberta_noseed": rf"roberta_processed/submission_roberta_full_noseed_processed.csv",
}

In [5]:
dfs = {}
for name, path in files.items():
	df = pd.read_csv(path)
	df = df.rename(columns={"Predicted": name})
	dfs[name] = df

# merge progressivo
df_all = None
for df in dfs.values():
	if df_all is None:
		df_all = df
	else:
		df_all = df_all.merge(df, on="Id")

print("Samples:", len(df_all))
df_all.head()


Samples: 20000


,Id,deberta_42,deberta_1337,deberta_2024,deberta_noseed,roberta_42,roberta_1337,roberta_2024,roberta_noseed
0,0,5,5,3,5,5,3,5,1
1,1,2,2,2,2,2,2,2,2
2,2,0,0,0,0,0,0,0,0
3,3,0,6,1,1,1,1,0,1
4,4,0,0,0,0,0,0,0,0


In [6]:
def agreement(col1, col2):
	return (df_all[col1] == df_all[col2]).mean()

pairs = list(itertools.combinations(dfs.keys(), 2))

results = []
for a, b in pairs:
	results.append({
		"model_a": a,
		"model_b": b,
		"agreement": agreement(a, b)
	})

agreement_df = pd.DataFrame(results).sort_values("agreement")
agreement_df


,model_a,model_b,agreement
21,deberta_noseed,roberta_noseed,0.85790
12,deberta_1337,roberta_noseed,0.86620
17,deberta_2024,roberta_noseed,0.86710
6,deberta_42,roberta_noseed,0.86730
19,deberta_noseed,roberta_1337,0.87115
18,deberta_noseed,roberta_42,0.87290
8,deberta_1337,deberta_noseed,0.87290
20,deberta_noseed,roberta_2024,0.87300
11,deberta_1337,roberta_2024,0.87435
2,deberta_42,deberta_noseed,0.87665


In [7]:
def intra(prefix):
	cols = [c for c in df_all.columns if c.startswith(prefix)]
	res = []
	for a, b in itertools.combinations(cols, 2):
		res.append({
			"pair": f"{a} vs {b}",
			"agreement": (df_all[a] == df_all[b]).mean()
		})
	return pd.DataFrame(res)

print("RoBERTa intra-seed")
display(intra("roberta"))

print("DeBERTa intra-seed")
display(intra("deberta"))


RoBERTa intra-seed


,pair,agreement
0,roberta_42 vs roberta_1337,0.91495
1,roberta_42 vs roberta_2024,0.91110
2,roberta_42 vs roberta_noseed,0.88715
3,roberta_1337 vs roberta_2024,0.90795
4,roberta_1337 vs roberta_noseed,0.88585
5,roberta_2024 vs roberta_noseed,0.88725


DeBERTa intra-seed


,pair,agreement
0,deberta_42 vs deberta_1337,0.89820
1,deberta_42 vs deberta_2024,0.89665
2,deberta_42 vs deberta_noseed,0.87665
3,deberta_1337 vs deberta_2024,0.89555
4,deberta_1337 vs deberta_noseed,0.87290
5,deberta_2024 vs deberta_noseed,0.87965


In [9]:
model_cols = list(dfs.keys())

df_all["unanimous"] = df_all[model_cols].nunique(axis=1) == 1
df_all["disagreement"] = ~df_all["unanimous"]

print("Unanimous:", df_all["unanimous"].mean().round(4))
print("Disagreement:", df_all["disagreement"].mean().round(4))


Unanimous: 0.7306
Disagreement: 0.2694


In [10]:
from collections import Counter

majority = df_all[model_cols].mode(axis=1)[0]

outliers = {}
for col in model_cols:
	outliers[col] = (df_all[col] != majority).mean()

pd.Series(outliers).sort_values()


roberta_42        0.06280
roberta_1337      0.06405
roberta_2024      0.06910
deberta_42        0.07740
deberta_2024      0.07755
deberta_1337      0.07760
roberta_noseed    0.08850
deberta_noseed    0.09345
dtype: float64